In [18]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split, WeightedRandomSampler
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from model.sparsemax import Sparsemax
import os

In [4]:
from torch.utils.data import Dataset
import pandas as pd
import scanpy as sc
import numpy as np
import torch
import scipy.sparse as sp
from scipy.spatial.distance import cdist
from tqdm import trange
from scipy.sparse import issparse


def preprocess_data(adata, immune_cell, n_genes, resolution):
    # Read the data

    # Ensure adata is not a view
    adata = adata.copy()
    adata.var_names_make_unique()  # Ensure unique gene names

    # Filter the tumor cells
    print(adata.obs['cell_type'].unique())
    tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

    # Debug: Check tumor cells
    print(f"Tumor cells shape after filtering: {tumor_cells.shape}")
    if tumor_cells.shape[0] == 0:
        print("Warning: No tumor cells found after filtering.")

    # Calculate mean expression
    if issparse(tumor_cells.X):
        # Convert to dense and compute mean expression
        mean_expression = np.asarray(tumor_cells.X.mean(axis=0)).ravel()
    else:
        mean_expression = tumor_cells.X.mean(axis=0)

    # Get gene names
    gene_names = tumor_cells.var_names

    # Select top n genes
    print(f"Selecting top {n_genes} genes based on mean expression")
    if n_genes > len(gene_names):
        n_genes = int(len(gene_names) * 0.2)
    top_n_gene_indices = mean_expression.argsort()[-n_genes:][::-1]
    top_n_gene_names = gene_names[top_n_gene_indices]
    print(f"Top {n_genes} genes: {top_n_gene_names}")

    # Subset adata using gene names to keep indices consistent
    adata = adata[:, top_n_gene_names].copy()

    adata.obs[immune_cell] = adata.obs[immune_cell].astype(float)
    tumor_cells.obs[immune_cell] = tumor_cells.obs[immune_cell].astype(float)

    # Binarize the immune cell column based on the percentile value if resolution is not 'high'
    if resolution != 'high':
        if tumor_cells.obs[immune_cell].empty:
            print(f"Error: tumor_cells.obs[{immune_cell}] is empty.")
        else:
            unique_values = tumor_cells.obs[immune_cell].unique()
            if set(unique_values).issubset({0, 1}):
                print(f"tumor_cells.obs[{immune_cell}] is already binary. Skipping binarization.")
            else:
                percentile_value = np.percentile(tumor_cells.obs[immune_cell], 75)
                print(f"Percentile value: {percentile_value}")
                adata.obs[immune_cell] = np.where(adata.obs[immune_cell] > percentile_value, 1, 0)
                print(f"adata.obs[{immune_cell}] after binarization: {adata.obs[immune_cell].head()}")

    return adata


class BagsDataset(Dataset):
    def __init__(self, input_data, immune_cell, max_instances=None, radius=200, resolution='low', n_genes=500, k=2):
        self.immune_cell = map_immune_cell(immune_cell)
        self.max_instances = max_instances
        self.radius = radius
        self.resolution = resolution
        self.n_genes = n_genes
        self.k = k  # Number of bags per batch
        if isinstance(input_data, str):
            self.batches = self.create_bags_from_csv(input_data)
        elif isinstance(input_data, sc.AnnData):
            input_data = preprocess_data(input_data, immune_cell, n_genes, self.resolution)
            print(f"Preprocessed data: {input_data.X.shape}")
            self.batches = self.create_bags_from_adata(input_data)
        else:
            raise ValueError("input_data must be either a path to a CSV file or an AnnData object")

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        batch = self.batches[idx]
        # batch is a list of bags
        batch_data = []
        for bag in batch:
            distances = bag['distances']
            gene_expression = bag['gene_expression']
            label = bag['label']
            core_idx = bag['core_idx']
            gene_names = bag['gene_names']
            cell_id = bag['cell_id']
            bag_dict = {
                'distances': distances,
                'gene_expression': gene_expression,
                'label': label,
                'core_idx': core_idx,
                'gene_names': gene_names,
                'cell_id': cell_id
            }
            batch_data.append(bag_dict)
        return batch_data

    def create_bags_from_csv(self, csv_file):
        data = pd.read_csv(csv_file)
        adata_radius_list = []
        for _, row in data.iterrows():
            adata_path = row['adata']
            print(f"Reading adata from {adata_path}")
            resolution = row['resolution'] if 'resolution' in row and not pd.isna(row['resolution']) else self.resolution
            adata = sc.read_h5ad(adata_path)
            adata.obs_names_make_unique()
            adata = preprocess_data(adata, self.immune_cell, self.n_genes, resolution=resolution)
            radius = row['radius'] if 'radius' in row and not pd.isna(row['radius']) else self.radius
            adata_radius_list.append((adata, radius, resolution))
            print(f"Processing: adata={adata_path.split('/')[-1]}, radius={radius}, resolution={resolution}")
        return self.create_bags(adata_radius_list)

    def create_bags_from_adata(self, adata):
        adata_radius_list = [(adata, self.radius, self.resolution)]
        return self.create_bags(adata_radius_list)

    def create_bags(self, adata_radius_list):
        all_batches = []
        for adata, radius, resolution in adata_radius_list:
            # Collect positive and negative bags per adata
            positive_bags = []
            negative_bags = []
            spatial_coords_x = adata.obs['X'].astype(float)
            spatial_coords_y = adata.obs['Y'].astype(float)
            spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
            gene_expression = adata.X
            labels = adata.obs[self.immune_cell].values.astype(int)  
            adata.obs['cell_type'] = adata.obs['cell_type'].astype(int)
            cell_types = adata.obs['cell_type'].values
            barcodes = adata.obs.index.values  
            gene_names = adata.var_names.tolist()

            for i in trange(len(spatial_coords), desc=f"Creating Bags with radius {radius}", ncols=100):
                if cell_types[i] == 0:
                    continue
                dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
                in_circle = np.where(dist_matrix_row <= radius)[0]
                in_circle = [idx for idx in in_circle if cell_types[idx] == 1]
                num_tumor_cells = len(in_circle)
                if resolution == 'high' and num_tumor_cells < 10:
                    continue
                if resolution == 'high':
                    in_circle = [idx for idx in in_circle if idx != i]
                if len(in_circle) == 0:
                    continue
                if self.max_instances is not None and len(in_circle) > self.max_instances:
                    continue

                gene_data = gene_expression[in_circle]
                distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

                bag = {
                    'distances': distances,
                    'gene_expression': gene_data,
                    'label': labels[i],
                    'core_idx': i,
                    'gene_names': gene_names,
                    'cell_id': barcodes[i]
                }

                if labels[i] == 1:
                    positive_bags.append(bag)
                else:
                    negative_bags.append(bag)

            num_negative_per_batch = self.k - 1
            if len(negative_bags) < num_negative_per_batch:
                print(f"Not enough negative bags in this adata to create batches. Dropping extra positive bags.")
                num_batches = len(negative_bags) // num_negative_per_batch
                if num_batches == 0:
                    continue 
                if len(positive_bags) > num_batches:
                    positive_bags = positive_bags[:num_batches]
            else:
                num_batches = min(len(positive_bags), len(negative_bags) // num_negative_per_batch)
                if len(positive_bags) > num_batches:
                    positive_bags = positive_bags[:num_batches]
                if len(negative_bags) > num_batches * num_negative_per_batch:
                    negative_bags = negative_bags[:num_batches * num_negative_per_batch]
        
            np.random.shuffle(negative_bags)

            for i in range(num_batches):
                batch = [positive_bags[i]] + negative_bags[i * num_negative_per_batch: (i + 1) * num_negative_per_batch]
                all_batches.append(batch)

        total_batches = len(all_batches)
        print(f"Total batches created: {total_batches}")
        return all_batches



def custom_collate_fn(batch):
    
    batch_bags = batch[0]
    distances_list = []
    gene_expressions_list = []
    labels_list = []
    core_idxs_list = []
    gene_names_list = []
    cell_ids_list = []
    for bag_data in batch_bags:
        distances = torch.tensor(bag_data['distances'], dtype=torch.float32)
        gene_expression = bag_data['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag_data['label'], dtype=torch.float32)
        core_idx = bag_data['core_idx']
        gene_names = bag_data['gene_names']
        cell_id = bag_data['cell_id']
        distances_list.append(distances)
        gene_expressions_list.append(gene_expression)
        labels_list.append(label)
        core_idxs_list.append(core_idx)
        gene_names_list.append(gene_names)
        cell_ids_list.append(cell_id)
    return distances_list, gene_expressions_list, labels_list, core_idxs_list, gene_names_list, cell_ids_list



def map_immune_cell(immune_cell):
    mapping = {
        'tcell': 'T',
        'bcell': 'B',
        'macrophage': 'Macrophage',
        'neutrophil': 'Neutrophil',
        'fibroblast': 'Fibroblast',
        'endothelial': 'Endothelial',
    }
    if immune_cell in mapping:
        return mapping[immune_cell]
    else:
        raise ValueError('Invalid immune cell type')


In [5]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F


class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x

class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.beta = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, distances_list, gene_expressions_list, current_genes_list):
        bag_outputs = []
        
        # Since each bag may have different genes, we process them individually
        for distances, gene_expression, current_genes in zip(distances_list, gene_expressions_list, current_genes_list):
            # Process distances and gene expressions through their respective networks
            distances = self.distance(distances)  # Shape: [num_instances, 1]
            gene_expression = self.gene_expression(gene_expression)  # Shape: [num_instances, num_genes]

            # Get immunogenicity vector and filtered genes for the current bag
            immunogenicity_vector, filtered_genes = self.immunogenicity(current_genes)
            
            if len(filtered_genes) == 0:
                continue  # Skip this bag if no overlapping genes
            
            # Map gene names to indices
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            
            # Select the relevant gene expressions
            gene_expression = gene_expression[:, gene_indices]  # Shape: [num_instances, num_filtered_genes]
            
            # Compute z as the dot product between gene expression and immunogenicity
            z = gene_expression @ immunogenicity_vector  # Shape: [num_instances]
            z = z.unsqueeze(1)  # Shape: [num_instances, 1]
            
            # Compute the bag output
            bag_output = distances * z  # Element-wise multiplication
            bag_output = torch.sum(bag_output, dim=0)  # Sum over instances
            bag_output = torch.exp(self.alpha) * bag_output + self.beta
            #print(bag_output)
            #bag_output = torch.sigmoid(bag_output)  # Final output for the bag
            
            bag_outputs.append(bag_output)
        
        if len(bag_outputs) == 0:
            return None  # Handle this case appropriately in your training loop
        
        # Stack outputs for all bags in the batch
        return torch.stack(bag_outputs).squeeze(dim=1)  # Shape: [batch_size]
    

class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [6]:
def load_all_genes(reference_gene_file):
    all_genes = pd.read_csv(reference_gene_file)
    return all_genes['Gene'].values.tolist()


In [7]:

def save_metrics(epoch, train_loss, val_loss, val_auroc,a,b,alpha,beta, output_dir):
    file_path = os.path.join(output_dir, 'training_metrics.csv')
    if not os.path.exists(file_path):
        # Create the CSV file with headers
        with open(file_path, 'w') as f:
            f.write('Epoch,Train Loss,Val Loss,Val AUROC,a,b,alpha,beta\n')
    
    # Append metrics for the current epoch
    with open(file_path, 'a') as f:
        f.write(f'{epoch},{train_loss},{val_loss},{val_auroc},{a},{b},{alpha},{beta}\n')


In [8]:
def save_ig_scores(epoch, all_genes, ig_scores_before_training, ig_scores_after_training, output_dir):
    # Create a DataFrame with IG scores before and after the current epoch
    ig_score_data = {
        'Gene': all_genes,
        'IG Score Before Training': ig_scores_before_training,
        'IG Score After Training': ig_scores_after_training,
    }
    df = pd.DataFrame(ig_score_data)
    
    df['Difference'] = df['IG Score After Training'] - df['IG Score Before Training']
    df = df.sort_values(by='Difference', ascending=False)

    output_path = os.path.join(output_dir, f'ig_score_changes_epoch_{epoch+1}.csv')
    df.to_csv(output_path, index=False)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [15]:
all_genes = load_all_genes('data/human_filtered.csv')
print('Reference genes loaded:', len(all_genes))

Reference genes loaded: 23182


In [16]:
model = MIL(all_genes).to(device)  # Adjust 'k' as needed
optimizer = optim.AdamW(model.parameters(), lr=0.1, weight_decay=0.01)

In [17]:
dataset = BagsDataset(
        'data/t_cell.csv',
        immune_cell='tcell',
        max_instances=500,
        n_genes=10000,
        k=4  # Ensure 'k' matches the number of negative bags per batch
    )

Reading adata from /project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancer/T_cell.h5ad
['0', '1']
Categories (2, object): ['0', '1']
Tumor cells shape after filtering: (1226, 17943)
Selecting top 10000 genes based on mean expression
Top 10000 genes: Index(['IGHM', 'IGLC1', 'IGKC', 'JCHAIN', 'B2M', 'TMSB4X', 'UBC', 'UBA52',
       'ACTB', 'FTL',
       ...
       'ADCY4', 'GPR75', 'KDM4A', 'MKX', 'MYPOP', 'NAV1', 'IZUMO4', 'UBE2L3',
       'LMOD1', 'CCDC153'],
      dtype='object', length=10000)
Percentile value: 4.840213894844055
adata.obs[T] after binarization: AAACAAGTATCTCCCA-1    0
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    0
AAACAGAGCGACTCCT-1    0
AAACAGCTTTCAGAAG-1    0
Name: T, dtype: int64
Processing: adata=T_cell.h5ad, radius=300, resolution=low
Reading adata from /project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancerWholeTranscriptome/T_cell.h5ad
['1', '0']
Categories (2, object): ['0', '1']
Tumor cells shap

/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[0. 1.]
Tumor cells shape after filtering: (2270, 33538)
Selecting top 10000 genes based on mean expression
Top 10000 genes: Index(['RPL19', 'MT-CO3', 'MT-CO1', 'MT-CO2', 'MT-ND4', 'S100A7', 'TMSB10',
       'MT-ND1', 'MT-ATP6', 'RPL23',
       ...
       'BHLHB9', 'TMOD1', 'DTD1', 'ENOX2', 'ABRAXAS1', 'PMS2', 'APOBEC3B',
       'HDHD2', 'ZNF232', 'TAZ'],
      dtype='object', length=10000)
tumor_cells.obs[T] is already binary. Skipping binarization.
Processing: adata=TumE2_preprocessed.h5ad, radius=300, resolution=low
Reading adata from /project/DPDS/Wang_lab/s439765/spatial_tcr/SpatialVDJ/SpatialVDJ_forZenodo/data/breast_cancer/TumC2_preprocessed.h5ad
[0. 1.]


/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Tumor cells shape after filtering: (1350, 33538)
Selecting top 10000 genes based on mean expression
Top 10000 genes: Index(['MT-CO1', 'MT-CO3', 'RPL19', 'MT-CO2', 'MT-ND4', 'TMSB10', 'MT-ND1',
       'MT-ND3', 'RPL41', 'MT-ATP6',
       ...
       'WDR35', 'CXCL11', 'CREBRF', 'DCAF4', 'EPG5', 'OTUD6B', 'MEIS2',
       'ZBED5-AS1', 'ARNTL', 'CCT6B'],
      dtype='object', length=10000)
tumor_cells.obs[T] is already binary. Skipping binarization.
Processing: adata=TumC2_preprocessed.h5ad, radius=300, resolution=low
Reading adata from /project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_TCR_scRNA/p16_preprocessed.h5ad


/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


[1 0]
Tumor cells shape after filtering: (621, 36601)
Selecting top 10000 genes based on mean expression
Top 10000 genes: Index(['MT-CO2', 'RPL41', 'MT2A', 'MT-CO3', 'S100A6', 'RPLP1', 'RPS18',
       'RPL11', 'RPL37A', 'RPS3',
       ...
       'TOGARAM1', 'PPP1R12B', 'STX17', 'KHK', 'RANBP6', 'CD37', 'FMNL1',
       'RFX1', 'MMP16', 'CLDN14'],
      dtype='object', length=10000)
tumor_cells.obs[T] is already binary. Skipping binarization.
Processing: adata=p16_preprocessed.h5ad, radius=300, resolution=low
Reading adata from /project/DPDS/Wang_lab/s439765/spatial_tcr/slide_tag/slide_tag_melanoma_processed.h5ad
[2. 0. 1.]
Tumor cells shape after filtering: (833, 36601)
Selecting top 10000 genes based on mean expression
Top 10000 genes: Index(['MALAT1', 'B2M', 'HLA-B', 'CD63', 'CALD1', 'NEAT1', 'MKLN1', 'EXOC4',
       'HLA-A', 'IFI6',
       ...
       'SLA', 'DCBLD1', 'HEATR3', 'TOR2A', 'LINC02732', 'VPS26B', 'AC012146.1',
       'PHLDB3', 'CHAF1A', 'GORASP1'],
      dtype='object', l

Creating Bags with radius 300: 100%|███████████████████████████| 2535/2535 [00:17<00:00, 145.62it/s]

Total batches created: 17666


In [21]:
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [22]:
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


In [24]:
num_epochs = 10
output_dir = 'test_output'

In [25]:
best_val_loss = float('inf')
best_model_path = os.path.join(output_dir, 'best_model.pth')

In [26]:
ig_scores_before_training = model.immunogenicity.ig.clone().detach().cpu()
ig_scores_before_training = [score.item() for score in ig_scores_before_training]

In [27]:
for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        # Lists to store outputs and labels for AUROC calculation
        all_outputs = []
        all_labels = []
        
        with tqdm(train_loader, unit="batch") as tepoch:
            for i, batch_data in enumerate(tepoch):
                tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")
                optimizer.zero_grad()

                distances_list, gene_expressions_list, labels_list, core_idxs_list, gene_names_list, cell_ids_list = batch_data
                
                distances_list = [distances.to(device) for distances in distances_list]
                gene_expressions_list = [gene_exp.to(device) for gene_exp in gene_expressions_list]
                labels = torch.stack(labels_list).float().to(device)
                current_genes_list = gene_names_list  
                
                outputs = model(distances_list, gene_expressions_list, current_genes_list)
                
                if outputs is None:
                    continue  # Skip this batch if the model returns None
                
                if outputs.shape[0] != labels.shape[0]:
                    # Handle mismatch in batch sizes if necessary
                    continue
                
                # Compute custom loss
                # Identify positive and negative outputs
                positive_idxs = (labels == 1).nonzero(as_tuple=True)[0]
                negative_idxs = (labels == 0).nonzero(as_tuple=True)[0]
                
                if positive_idxs.numel() == 0 or negative_idxs.numel() == 0:
                    continue  # Skip batch if no positive or negative bags
                
                positive_output = outputs[positive_idxs]
                negative_outputs = outputs[negative_idxs]
                
                # Compute mean of negative outputs and loss
                mean_negative_output = negative_outputs.mean()
                positive_output = positive_output.mean() 
                
                loss = torch.relu(mean_negative_output - positive_output+0.1)
                loss.backward()
                optimizer.step()
        
                running_loss += loss.item()
                tepoch.set_postfix(loss=loss.item())
                
                all_outputs.extend(outputs.detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        
        # Compute Training AUROC
        try:
            epoch_auc = roc_auc_score(all_labels, all_outputs)
        except ValueError:
            epoch_auc = float('nan')  # Handle case where AUROC can't be computed
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, AUROC: {epoch_auc:.4f}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_all_outputs = []
        val_all_labels = []
        with torch.no_grad():
            with tqdm(val_loader, unit="batch") as vtepoch:
                for val_batch_data in vtepoch:
                 
                    val_distances_list, val_gene_expressions_list, val_labels_list, val_core_idxs_list, val_gene_names_list, val_cell_ids_list = val_batch_data
                    
                    
                    val_distances_list = [distances.to(device) for distances in val_distances_list]
                    val_gene_expressions_list = [gene_exp.to(device) for gene_exp in val_gene_expressions_list]
                    val_labels = torch.stack(val_labels_list).float().to(device)
                    val_current_genes_list = val_gene_names_list  # List of gene names for each bag
                    
                    # Forward pass
                    val_outputs = model(val_distances_list, val_gene_expressions_list, val_current_genes_list)
                    
                    if val_outputs is None:
                        continue  # Skip this batch if the model returns None
                    
                    if val_outputs.shape[0] != val_labels.shape[0]:
                        # Handle mismatch in batch sizes if necessary
                        continue
                    
                    # Compute custom loss
                    # Identify positive and negative outputs
                    positive_idxs = (val_labels == 1).nonzero(as_tuple=True)[0]
                    negative_idxs = (val_labels == 0).nonzero(as_tuple=True)[0]
                    
                    if positive_idxs.numel() == 0 or negative_idxs.numel() == 0:
                        continue 
                    
                    positive_output = val_outputs[positive_idxs]
                    negative_outputs = val_outputs[negative_idxs]
                    
                    # Compute mean of negative outputs and loss
                    mean_negative_output = negative_outputs.mean()
                    positive_output = positive_output.mean()
                    
                    loss = torch.relu(mean_negative_output - positive_output+0.1)
                    val_loss += loss.item()
                    vtepoch.set_postfix(val_loss=loss.item())
                    
                    # Accumulate outputs and labels for AUROC calculation
                    val_all_outputs.extend(val_outputs.detach().cpu().numpy())
                    val_all_labels.extend(val_labels.cpu().numpy())
            
            val_loss /= len(val_loader)
            
            # Compute Validation AUROC
            try:
                val_epoch_auc = roc_auc_score(val_all_labels, val_all_outputs)
            except ValueError:
                val_epoch_auc = float('nan')  # Handle case where AUROC can't be computed
            
            print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_epoch_auc:.4f}')

        # Save the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved with validation loss {val_loss:.4f}")
        
        torch.save(model.state_dict(), os.path.join(output_dir, f'model_epoch_{epoch+1}.pth'))
        
        a = model.distance.a.clone().detach().cpu().item()
        b = model.gene_expression.b.clone().detach().cpu().item()
        alpha =model.alpha.clone().detach().cpu().item()
        beta = model.beta.clone().detach().cpu().item()
        
        # Save metrics
        save_metrics(epoch+1, train_loss, val_loss, val_epoch_auc,
                     a, b, alpha, beta,
                     output_dir)

        # Save IG scores after each epoch
        ig_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()
        ig_scores_after_training = [score.item() for score in ig_scores_after_training]
        save_ig_scores(epoch, all_genes, ig_scores_before_training, ig_scores_after_training, output_dir)

Epoch 1/10:  20%|██        | 2475/12366 [02:04<08:16, 19.93batch/s, loss=0.0755]  


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), os.path.join(output_dir, 'final_model.pth'))
